# Step-1 : Business Problem Understanding

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Step-2 : Data Preprocessing

In [51]:
df = pd.read_excel('insurance.xlsx')
df

,age,sex,bmi,children,smoker,region,expenses
0,19,female,27.9,0,yes,southwest,16884.92
1,18,male,33.8,1,no,southeast,1725.55
2,28,male,33.0,3,no,southeast,4449.46
3,33,male,22.7,0,no,northwest,21984.47
4,32,male,28.9,0,no,northwest,3866.86
...,...,...,...,...,...,...,...
1333,50,male,31.0,3,no,northwest,10600.55
1334,18,female,31.9,0,no,northeast,2205.98
1335,18,female,36.9,0,no,southeast,1629.83
1336,21,female,25.8,0,no,southwest,2007.95


In [52]:
df.head()

,age,sex,bmi,children,smoker,region,expenses
0,19,female,27.9,0,yes,southwest,16884.92
1,18,male,33.8,1,no,southeast,1725.55
2,28,male,33.0,3,no,southeast,4449.46
3,33,male,22.7,0,no,northwest,21984.47
4,32,male,28.9,0,no,northwest,3866.86


In [53]:
df.shape

(1338, 7)

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   expenses  1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


### **Exploratory Data Analysis**

In [55]:
categorical=[]
continous=[]
check =[]

d_types = dict(df.dtypes)
for name, dtype in d_types.items():
    if str(dtype) == 'object':
        categorical.append(name)
    elif str(dtype) == 'float64':
        continous.append(name)
    else:
        check.append(name)

print('categorical features:', categorical)
print('continous features:', continous)
print('features to be checked:', check)

categorical features: ['sex', 'smoker', 'region']
continous features: ['bmi', 'expenses']
features to be checked: ['age', 'children']


In [56]:
d_types = dict(df.dtypes)
for name, type_ in d_types.items():
    if str(type_) == 'object':
        print(f'<====={name}=====>')
        print(df[name].value_counts())

<=====sex=====>
sex
male      676
female    662
Name: count, dtype: int64
<=====smoker=====>
smoker
no     1064
yes     274
Name: count, dtype: int64
<=====region=====>
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64


In [57]:
df.describe()

,age,bmi,children,expenses
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.665471,1.094918,13270.422414
std,14.049960,6.098382,1.205493,12110.011240
min,18.000000,16.000000,0.000000,1121.870000
25%,27.000000,26.300000,0.000000,4740.287500
50%,39.000000,30.400000,1.000000,9382.030000
75%,51.000000,34.700000,2.000000,16639.915000
max,64.000000,53.100000,5.000000,63770.430000


## **Data Preprocessing**

In [58]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
expenses    0
dtype: int64

In [59]:
# drop th region column, 
df.drop('region', axis=1, inplace=True)

In [60]:
# encoding the sex column
df['sex'].replace({'female':0, 'male':1}, inplace=True)

# encoding 'smoker' column
df['smoker'].replace({'no':0, 'yes':1}, inplace=True)

## **X & y**

In [61]:
X = df.drop('expenses', axis=1)
y = df['expenses']

In [62]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9)

# Step-4 : Modelling & evalution
**Lasso Regression with default parameters**

In [63]:
# Modelling
from sklearn.linear_model import ElasticNet
enr_base = ElasticNet()
enr_base.fit(X_train, y_train)

#Predictionsbase
train_predictions = enr_base.predict(X_train)
test_predictions = enr_base.predict(X_test)

#Evalutions
print('Train R2:',enr_base.score(X_train, y_train))
print('Test R2:',enr_base.score(X_test, y_test))
from sklearn.model_selection import cross_val_score
print("Cross Validation Score:", cross_val_score(enr_base, X, y, cv=5).mean())

Train R2: 0.39155822533558715
Test R2: 0.39702776413473506
Cross Validation Score: 0.3889250431216654


**Applying Hyperparameter tunning for Lasso Regression**

In [64]:
from sklearn.model_selection import GridSearchCV

#Model
estimator = ElasticNet()

#Parameters & Values
param_grid = {'alpha':[0.1,0.2,1,2,3,5,10],'l1_ratio':[0.1,0.5,0.75,0.9,0.95,1]}

#Identifying the best vale of of the parameter within given values for the given data
model_hp = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_mean_squared_error')
model_hp.fit(X_train, y_train)
model_hp.best_params_

{'alpha': 10, 'l1_ratio': 1}

**Rebuilt Lasso Model using best hyperparameter**

In [65]:
# Modelling
enr_best = ElasticNet(alpha=10, l1_ratio=1)
enr_best.fit(X_train, y_train)

print('Intercept:', enr_best.intercept_)
print('coefficients:', enr_best.coef_)

# Predictions
train_predictions = enr_best.predict(X_train)
test_predictions = enr_best.predict(X_test)

# Evalutions
print('Train R2:', enr_best.score(X_train, y_train))
print('Test R2:', enr_best.score(X_test, y_test))
print('Cross Validation Score:', cross_val_score(enr_best, X, y, cv=5).mean())

Intercept: -11449.28756082979
coefficients: [ 2.56838444e+02 -6.43158858e-01  3.04860929e+02  4.34656692e+02
  2.35631810e+04]
Train R2: 0.7433083585849637
Test R2: 0.7755411716841649
Cross Validation Score: 0.7467299170217538


### **Prediction on New Data**

In [66]:
input_data = {'age':31,
             'sex':'female',
             'bmi':25.74,
             'children':0,
             'smoker':'no',
             'region':'northeast'}

**Preprocessing the Data**

In [67]:
df_test = pd.DataFrame(input_data, index=[0])

df_test.drop('region', axis=1, inplace=True)
df_test['sex'].replace({'female':0, 'male':1}, inplace=True)
df_test['smoker'].replace({'no':0, 'yes':1}, inplace=True)

transormed_data = df_test

### **Predict**

In [68]:
enr_best.predict(transormed_data)

array([4359.82451623])